# Short-Horizon Trade Return Modeling

**Data**
- `train` — 90,000 rows. Features `0`–`65` are anonymized; `TARGET` is the realized return of a short-horizon trade.
- `test` — 19,700 rows that come after train in time. **Locked:** you need a password to decrypt it (see the bottom of the notebook).

Rows are in time order, but the time steps between them are **not uniform**.

**Task:** Build a model that decides which trades to take. Evaluate it out of sample with time-ordered validation on train. Choose a trade-selection rule. Unlock the test set **only once**, for the final evaluation.

**Setup:** the next cell downloads `train.csv.gz` and `test.enc` from GitHub. No sign-in needed.

In [ ]:
# Download the data files from GitHub
import io, urllib.request

REPO = 'https://github.com/victorwangse/witg-qis/raw/main/'
for name in ['train.csv.gz', 'test.enc']:
    urllib.request.urlretrieve(REPO + name, name)
    print('downloaded', name)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

train_df = pd.read_csv('train.csv.gz')
print(train_df.shape)
train_df.head()

In [ ]:
train_df['TARGET'].describe()

## Your work

EDA → features → model → time-series CV → trade-selection threshold.

---
## Locked test set

Run this **only** after your model and threshold are frozen. It asks for the password.

In [ ]:
from getpass import getpass
import gzip
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
from cryptography.hazmat.primitives import hashes
from cryptography.exceptions import InvalidTag

def unlock_test(path='test.enc'):
    blob = open(path, 'rb').read()
    assert blob[:4] == b'ENC1', 'not an encrypted test file'
    salt, nonce, ct = blob[4:20], blob[20:32], blob[32:]
    pw = getpass('Test-set password: ')
    key = PBKDF2HMAC(hashes.SHA256(), 32, salt, 600_000).derive(pw.encode())
    try:
        raw = AESGCM(key).decrypt(nonce, ct, b'test_v1')
    except InvalidTag:
        raise ValueError('Wrong password.') from None
    return pd.read_csv(io.BytesIO(gzip.decompress(raw)))

test_df = unlock_test()
print(test_df.shape)